# Chapter 5: Evaluation Gates

## How to Stop a Bad Model Before It Reaches Your Client

Training is done. We have a model that looks at customer data and outputs a probability of churn. But how do we know if it's any good?

Before a restaurant serves a dish, the chef tastes it. If it's not up to standard, it doesn't go to the table. Our **evaluation gate** is that taste test:

- The model must prove it can sort customers by churn risk better than random chance
- If it fails, it gets rejected — no bad predictions reach the client
- If it passes, it goes into the model registry for production use

The key question: **what metric do we use as the gate?**

In [ ]:
import numpy as np
from churn_pipeline.steps.evaluation import evaluate_model, generate_model_card

## Why AUC-ROC, Not Accuracy?

**Accuracy is misleading with imbalanced data.** If only 15% of customers churn, a model that always predicts "won't churn" gets 85% accuracy — but catches zero actual leavers. Useless.

**AUC-ROC measures ranking quality.** Imagine sorting all customers by how likely the model thinks they'll leave:

- A perfect model puts ALL actual leavers at the top → AUC = 1.0
- A random model mixes leavers and stayers randomly → AUC = 0.5
- Our minimum: AUC = 0.70 → actual leavers are meaningfully higher in the list

**The gate rule:** pass/fail depends SOLELY on AUC ≥ 0.70. Other metrics (F1, precision, recall) are logged for information but don't affect the decision.

In [ ]:
# Simulate a GOOD model (clear separation between churners and stayers)
np.random.seed(42)
n = 200
y_true = np.array([0] * 160 + [1] * 40)  # 20% churn rate

# Good model: churners get high probs, stayers get low probs
y_pred_good = np.where(
    y_true == 1,
    np.random.uniform(0.6, 0.95, size=n),   # Churners → high
    np.random.uniform(0.05, 0.4, size=n),   # Stayers → low
)

result_good = evaluate_model(y_true, y_pred_good)

print("GOOD MODEL:")
print(f"  AUC-ROC:   {result_good.auc_roc:.4f}")
print(f"  F1:        {result_good.f1_score:.4f}")
print(f"  Precision: {result_good.precision:.4f}")
print(f"  Recall:    {result_good.recall:.4f}")
print(f"  PASSED:    {result_good.passed} ✓")

In [ ]:
# Simulate a BAD model (predictions are basically random noise)
y_pred_bad = np.random.uniform(0.3, 0.7, size=n)  # All near 0.5 → random

result_bad = evaluate_model(y_true, y_pred_bad)

print("BAD MODEL (random predictions):")
print(f"  AUC-ROC:   {result_bad.auc_roc:.4f}")
print(f"  F1:        {result_bad.f1_score:.4f}")
print(f"  Precision: {result_bad.precision:.4f}")
print(f"  Recall:    {result_bad.recall:.4f}")
print(f"  PASSED:    {result_bad.passed} ✗ — REJECTED")

## The Four Metrics Explained

| Metric | Question It Answers | Formula |
|--------|-------------------|--------|
| **AUC-ROC** | How well does the model sort customers? | Area under the ROC curve |
| **Precision** | Of everyone I flagged, how many actually left? | TP / (TP + FP) |
| **Recall** | Of everyone who left, how many did I catch? | TP / (TP + FN) |
| **F1** | Balance between precision and recall | 2 × (P × R) / (P + R) |

There's always a trade-off:
- **High precision, low recall:** "I'm very careful — I only flag customers I'm really sure about, but I miss many actual leavers"
- **High recall, low precision:** "I catch almost every leaver, but I also falsely flag many loyal customers"
- **F1** balances both — it's high only when both precision AND recall are good

In [ ]:
# Demonstrate the boundary: AUC exactly 0.70 passes
y_true_boundary = np.array([0] * 100 + [1] * 100)
y_pred_boundary = np.zeros(200)
y_pred_boundary[100:] = 0.8     # All positives at 0.8
y_pred_boundary[:70] = 0.3      # 70% of negatives below
y_pred_boundary[70:100] = 0.9   # 30% of negatives above

result_boundary = evaluate_model(y_true_boundary, y_pred_boundary)
print(f"Boundary test: AUC = {result_boundary.auc_roc} → passed = {result_boundary.passed}")
print(f"  (exactly 0.70 passes because the rule is >= not >)")

## The Model Card: A Nutrition Label for Your Model

When a model passes the gate, we generate a **model card** — all the metadata someone needs to understand what this model is, how it was trained, and how well it performs.

Think of it like a nutrition label on food: you can quickly assess if it's suitable for your needs without opening the container.

In [ ]:
import json

card = generate_model_card(
    eval_result=result_good,
    training_params={"max_depth": 6, "eta": 0.1, "subsample": 0.8},
    dataset_info={"client_id": "telco_ibm", "rows": 7043, "training_date": "2024-01-15"},
    feature_list=["tenure_months", "monthly_charges", "total_charges",
                   "contract_type", "support_tickets", "interaction_charges_x_tenure"],
    model_id="churn-xgb-v3",
)

print("Model Card (stored as JSON alongside the model artifact):")
print("=" * 60)
print(json.dumps(card, indent=2))

## Key Takeaways

1. **AUC-ROC is the gate metric** — it measures ranking quality, not accuracy
2. **Minimum AUC = 0.70** — below this, the model is too close to random
3. **Only AUC controls the gate** — F1, precision, recall are logged but don't block
4. **Model cards provide transparency** — every model gets a nutrition label
5. **The gate prevents bad models from reaching clients** — quality over speed

Next: Chapter 6 covers drift monitoring — how we detect when the world changes and the model may become unreliable.

---

*Source code: `src/churn_pipeline/steps/evaluation.py`*  
*Tests: `tests/property/test_evaluation.py`, `tests/unit/test_evaluation.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*